# Video-Image Conversion Utility

This notebook provides utilities to:
1. Convert a sequence of images (PNG, JPG, etc.) to an MP4 video
2. Extract frames from an MP4 video as individual image files

You can specify the direction (images-to-video or video-to-images) and the input file or folder path.

In [2]:
import os
import cv2
import glob
import argparse
import numpy as np
from tqdm import tqdm
from pathlib import Path

## Helper Functions

In [3]:
def images_to_video(image_folder, output_video_path, fps=30, image_format='png'):
    """
    Convert a sequence of images to a video file.
    
    Args:
        image_folder (str): Path to the folder containing image sequences
        output_video_path (str): Path where the output video will be saved
        fps (int): Frames per second for the output video
        image_format (str): Format of input images (png, jpg, etc.)
        
    Returns:
        bool: True if successful, False otherwise
    """
    # Ensure output directory exists
    os.makedirs(os.path.dirname(os.path.abspath(output_video_path)), exist_ok=True)
    
    # Get all images in the folder with the specified format
    images = sorted(glob.glob(os.path.join(image_folder, f'*.{image_format}')))
    
    if not images:
        print(f"No {image_format} images found in {image_folder}")
        return False
    
    # Read the first image to get dimensions
    frame = cv2.imread(images[0])
    h, w, _ = frame.shape
    
    # Define the codec and create VideoWriter object
    fourcc = cv2.VideoWriter_fourcc(*'mp4v')  # Use mp4v codec for MP4 files
    out = cv2.VideoWriter(output_video_path, fourcc, fps, (w, h))
    
    # Process all images
    print(f"Converting {len(images)} images to video...")
    for img_path in tqdm(images):
        frame = cv2.imread(img_path)
        out.write(frame)
    
    # Release the video writer
    out.release()
    print(f"Video saved to {output_video_path}")
    return True


def video_to_images(video_path, output_folder, image_format='png'):
    """
    Extract frames from a video file and save them as individual images.
    
    Args:
        video_path (str): Path to the input video file
        output_folder (str): Path to the folder where images will be saved
        image_format (str): Format to save images (png, jpg, etc.)
        
    Returns:
        bool: True if successful, False otherwise
    """
    # Ensure output directory exists
    os.makedirs(output_folder, exist_ok=True)
    
    # Open the video file
    cap = cv2.VideoCapture(video_path)
    
    if not cap.isOpened():
        print(f"Error opening video file: {video_path}")
        return False
    
    # Get video properties
    frame_count = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    fps = cap.get(cv2.CAP_PROP_FPS)
    
    print(f"Video information:")
    print(f"- Total frames: {frame_count}")
    print(f"- FPS: {fps}")
    
    # Extract frames
    print(f"Extracting frames to {output_folder}...")
    frame_idx = 0
    
    with tqdm(total=frame_count) as pbar:
        while True:
            ret, frame = cap.read()
            if not ret:
                break
                
            # Save frame as an image
            output_path = os.path.join(output_folder, f"frame_{frame_idx:06d}.{image_format}")
            cv2.imwrite(output_path, frame)
            
            frame_idx += 1
            pbar.update(1)
    
    # Release the video capture
    cap.release()
    print(f"Extracted {frame_idx} frames to {output_folder}")
    return True

## Command-line Interface Function

This function mimics the command-line interface if you want to run this as a Python script

In [4]:
def convert(direction, input_path, output_path, fps=30, image_format='png'):
    """
    Convert between video and image sequences.
    
    Args:
        direction (str): 'images_to_video' or 'video_to_images'
        input_path (str): Path to input video file or folder containing images
        output_path (str): Path to output video file or folder for extracted images
        fps (int): Frames per second (for images_to_video)
        image_format (str): Format of images (png, jpg, etc.)
        
    Returns:
        bool: True if successful, False otherwise
    """
    if direction == 'images_to_video':
        return images_to_video(input_path, output_path, fps, image_format)
    elif direction == 'video_to_images':
        return video_to_images(input_path, output_path, image_format)
    else:
        print(f"Invalid direction: {direction}. Use 'images_to_video' or 'video_to_images'.")
        return False

## Example Usage

Here are examples of how to use the conversion functions:

In [6]:
# Example 1: Convert images to video
# Uncomment and modify the paths as needed

data_collection_nr = 1
for data_collection_nr in range(2, 3):
    print(f"Creating Videos for data collection: chb/subsequence{data_collection_nr}")

    for i in range(1, 5):
        cam_name = f'gopro{i}'
        image_folder = f'inputs/chb/subsequence{data_collection_nr}/{cam_name}'
        output_video = f'inputs/chb/chb{data_collection_nr}/{cam_name}.mp4'
        convert('images_to_video', image_folder, output_video, fps=30, image_format='png')
    cam_name = 'ORX_camera'
    image_folder = f'inputs/chb/subsequence{data_collection_nr}/{cam_name}'
    output_video = f'inputs/chb/chb{data_collection_nr}/{cam_name}.mp4'
    convert('images_to_video', image_folder, output_video, fps=30, image_format='png')

Creating Videos for data collection: chb/subsequence2
Converting 300 images to video...


100%|██████████| 300/300 [00:44<00:00,  6.69it/s]


Video saved to inputs/chb/chb2/gopro1.mp4
Converting 300 images to video...


100%|██████████| 300/300 [00:45<00:00,  6.54it/s]


Video saved to inputs/chb/chb2/gopro2.mp4
Converting 300 images to video...


...

Video saved to inputs/chb/chb2/gopro3.mp4
Converting 300 images to video...


100%|██████████| 300/300 [00:43<00:00,  6.88it/s]


Video saved to inputs/chb/chb2/gopro4.mp4
Converting 300 images to video...


100%|██████████| 300/300 [00:16<00:00, 18.13it/s]

Video saved to inputs/chb/chb2/ORX_camera.mp4


In [ ]:
# Example 2: Extract frames from a video
# Uncomment and modify the paths as needed

# video_path = 'path/to/input/video.mp4'
# output_folder = 'path/to/output/frames'
# convert('video_to_images', video_path, output_folder, image_format='png')

## Interactive Widget for Conversion

This interactive widget allows you to choose the conversion direction and provide input/output paths.

In [ ]:
import ipywidgets as widgets
from IPython.display import display, clear_output

def run_conversion(b):
    clear_output()
    display(direction_dropdown, input_text, output_text, fps_slider, format_dropdown, convert_button)
    
    direction = direction_dropdown.value
    input_path = input_text.value.strip()
    output_path = output_text.value.strip()
    fps = fps_slider.value
    image_format = format_dropdown.value
    
    if not input_path or not output_path:
        print("Error: Input and output paths are required")
        return
    
    print(f"Running conversion: {direction}")
    print(f"Input: {input_path}")
    print(f"Output: {output_path}")
    print(f"FPS: {fps}")
    print(f"Image format: {image_format}")
    print("\n")
    
    convert(direction, input_path, output_path, fps, image_format)

# Create widgets
direction_dropdown = widgets.Dropdown(
    options=['images_to_video', 'video_to_images'],
    value='images_to_video',
    description='Direction:',
    style={'description_width': 'initial'}
)

input_text = widgets.Text(
    value='',
    placeholder='Path to input folder or video file',
    description='Input path:',
    style={'description_width': 'initial'}
)

output_text = widgets.Text(
    value='',
    placeholder='Path to output video file or folder',
    description='Output path:',
    style={'description_width': 'initial'}
)

fps_slider = widgets.IntSlider(
    value=30,
    min=1,
    max=60,
    step=1,
    description='FPS:',
    disabled=False,
    continuous_update=False,
    orientation='horizontal',
    readout=True,
    readout_format='d'
)

format_dropdown = widgets.Dropdown(
    options=['png', 'jpg', 'jpeg', 'bmp', 'tiff'],
    value='png',
    description='Image format:',
    style={'description_width': 'initial'}
)

convert_button = widgets.Button(
    description='Convert',
    button_style='primary', 
    tooltip='Click to start conversion'
)

convert_button.on_click(run_conversion)

# Display the widgets
display(direction_dropdown, input_text, output_text, fps_slider, format_dropdown, convert_button)

## Python Script Version

The same functionality can be used as a Python script. Below is the equivalent script that can be run from the command line:

In [ ]:
# This is a code sample showing how the above functions would be used in a Python script

'''
#!/usr/bin/env python
import os
import cv2
import glob
import argparse
import numpy as np
from tqdm import tqdm
from pathlib import Path

# Function definitions here (images_to_video, video_to_images, convert)
# ...

if __name__ == "__main__":
    parser = argparse.ArgumentParser(description="Convert between video and image sequences")
    parser.add_argument(
        "direction",
        choices=["images_to_video", "video_to_images"],
        help="Conversion direction"
    )
    parser.add_argument(
        "input_path",
        help="Path to input video file or folder containing images"
    )
    parser.add_argument(
        "output_path",
        help="Path to output video file or folder for extracted images"
    )
    parser.add_argument(
        "--fps", 
        type=int, 
        default=30,
        help="Frames per second (for images_to_video)"
    )
    parser.add_argument(
        "--format", 
        default="png",
        help="Format of images (png, jpg, etc.)"
    )
    
    args = parser.parse_args()
    
    convert(args.direction, args.input_path, args.output_path, args.fps, args.format)
'''